<a href="https://colab.research.google.com/github/joshithasai2007/IBM-z-Datathon-Code/blob/main/IBM_codeipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch pydantic

In [2]:
from transformers import pipeline

print("Loading AI triage model... (this may take a minute)")
# Using a fast zero-shot classification model
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)
print("Model loaded successfully!")

Loading AI triage model... (this may take a minute)


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Model loaded successfully!


In [3]:
def triage_emergency_call(raw_transcript: str):
    # 1. Classify Urgency Level
    urgency_labels = ["Critical / Life-Threatening", "Urgent / Potential Injury", "Non-Emergency / Property Damage"]
    urgency_result = classifier(raw_transcript, candidate_labels=urgency_labels)

    top_urgency = urgency_result["labels"][0]
    urgency_score = urgency_result["scores"][0]

    # Map to dispatch priority levels
    if "Critical" in top_urgency:
        priority = "🔴 Priority 1 (Immediate Dispatch - Police/Fire/EMS)"
    elif "Urgent" in top_urgency:
        priority = "🟡 Priority 2 (Urgent Response - EMS/Police)"
    else:
        priority = "🟢 Priority 3 (Standard Dispatch)"

    # 2. Classify Incident Type
    category_labels = ["Medical Emergency", "Fire Outbreak", "Crime / Violence", "Traffic Accident", "Natural Disaster"]
    category_result = classifier(raw_transcript, candidate_labels=category_labels)
    top_category = category_result["labels"][0]

    # Print structured output for judges
    print(f"\n--- EMERGENCY TRIAGE REPORT ---")
    print(f"Transcript: \"{raw_transcript}\"")
    print(f"Detected Category: {top_category}")
    print(f"Assigned Urgency: {priority}")
    print(f"Confidence Score: {round(urgency_score * 100, 2)}%")
    print(f"-------------------------------")

In [4]:
# Test Case 1: Life-threatening
triage_emergency_call("There is a massive fire breaking out at 42 Wallaby Way! Someone is trapped inside and screaming!")

# Test Case 2: Non-emergency
triage_emergency_call("Someone parked their car blocking my driveway, can someone look into it?")


--- EMERGENCY TRIAGE REPORT ---
Transcript: "There is a massive fire breaking out at 42 Wallaby Way! Someone is trapped inside and screaming!"
Detected Category: Fire Outbreak
Assigned Urgency: 🟡 Priority 2 (Urgent Response - EMS/Police)
Confidence Score: 50.75%
-------------------------------

--- EMERGENCY TRIAGE REPORT ---
Transcript: "Someone parked their car blocking my driveway, can someone look into it?"
Detected Category: Crime / Violence
Assigned Urgency: 🟢 Priority 3 (Standard Dispatch)
Confidence Score: 42.92%
-------------------------------
